# Fiche 4 — Un client qui ne plante pas : timeout, quota, nouvelles tentatives

| | |
|---|---|
| 🎯 **Objectif** | Protéger une boucle d'appels contre les trois accidents courants : serveur muet, quota dépassé, panne passagère |
| ⏱️ **Durée** | 45 min |
| 🧰 **Prérequis** | Fiche 3 |
| 🔗 **Dans le projet** | Étape 4 (points de vigilance : timeouts, limites de requêtes) |

> **L'idée clé.** Sur 50 images, tout se passe souvent bien. Sur 500 000, une panne **arrivera**. Un bon client sait **attendre**, **réessayer ce qui peut réussir** et **abandonner vite ce qui ne réussira jamais**.

## 1. Un faux serveur pour s'entraîner sans réseau

Pour voir les accidents sans dépendre d'Internet, on fabrique un serveur imaginaire qui renvoie, dans l'ordre, les
codes qu'on lui donne. Il compte les appels reçus.

In [1]:
class FauxServeur:
    """Rejoue une suite de codes HTTP ; chaque appel « coûte » 0,5 s sur une horloge simulée."""

    def __init__(self, codes):
        self.codes, self.appels, self.horloge = list(codes), 0, 0.0

    def appeler(self):
        self.appels += 1
        self.horloge += 0.5
        return self.codes.pop(0) if self.codes else 200

    def attendre(self, secondes):
        self.horloge += secondes

## 2. Réessayer avec une attente qui double

Un 503 (« modèle en cours de chargement ») ou un 429 (« trop de requêtes ») disparaissent si on **patiente**.
On attend 2 s, puis 4, puis 8 : c'est l'*attente exponentielle* (*exponential backoff*). Réessayer tout de suite
ne ferait qu'ajouter de la charge à un serveur déjà débordé.

In [2]:
A_REESSAYER = {429, 500, 502, 503, 504}


def appel_robuste(serveur, essais_max=5):
    for essai in range(essais_max):
        code = serveur.appeler()
        if code == 200:
            return f"succès à l'essai {essai + 1}"
        if code not in A_REESSAYER:
            return f"abandon immédiat : code {code}, réessayer ne changera rien"
        serveur.attendre(2 * 2**essai)  # 2 s, 4 s, 8 s…
    return f"échec après {essais_max} essais"


serveur = FauxServeur([503, 503, 200])
print(appel_robuste(serveur), f"| {serveur.appels} appels | {serveur.horloge:.1f} s écoulées")

succès à l'essai 3 | 3 appels | 7.5 s écoulées


## 3. Le timeout : ne jamais attendre pour toujours

Sans `timeout`, `requests.post` peut rester bloqué indéfiniment si le serveur ne répond pas : la boucle de 50 images
se fige à la 12ᵉ, sans message. Avec `timeout=30`, `requests` lève `requests.Timeout` au bout de 30 s, qu'on traite
comme une erreur passagère (on réessaie).

```python
response = requests.post(URL, headers=headers, data=data, timeout=30)
```

## 4. Le quota : jamais plus de N requêtes par minute

Plutôt que d'attendre les 429 du serveur, on se limite soi-même. La règle : sur **n'importe quelle** fenêtre de 60 s,
au plus N appels. On garde l'heure des derniers appels ; si la fenêtre est pleine, on attend que le plus ancien en
sorte. Le package du projet le fait avec `api.RateLimiter` ; on le teste ici sur une horloge simulée.

In [3]:
from segvet.api import RateLimiter

horloge = {"t": 0.0}
limiteur = RateLimiter(3, clock=lambda: horloge["t"], sleep=lambda s: horloge.__setitem__("t", horloge["t"] + s))
instants = []
for _ in range(7):
    limiteur.wait()
    instants.append(horloge["t"])
    horloge["t"] += 1  # chaque appel dure 1 s
print("Appels partis à (s) :", instants)

Appels partis à (s) : [0.0, 1.0, 2.0, 60.0, 61.0, 62.0, 120.0]


**Lecture.** Les 3 premiers appels partent tout de suite ; le 4ᵉ attend que le 1ᵉʳ sorte de la fenêtre de 60 s. Le serveur ne voit jamais plus de 3 appels par minute.

## ⚠️ Piège : tout réessayer, même ce qui ne peut pas réussir

Une boucle « je réessaie tant que ça ne marche pas » paraît prudente. Avec un token faux (401), elle insiste
pour rien, attend de plus en plus longtemps… et peut faire bloquer le compte pour tentatives répétées.

In [4]:
def appel_naif(serveur, essais_max=5):
    for essai in range(essais_max):
        if serveur.appeler() == 200:
            return "succès"
        serveur.attendre(2 * 2**essai)
    return "échec"


for fonction in (appel_naif, appel_robuste):
    serveur = FauxServeur([401] * 10)
    resultat = fonction(serveur)
    print(f"{fonction.__name__:<14}: {resultat:<55} | {serveur.appels} appels, {serveur.horloge:.0f} s perdues")

appel_naif    : échec                                                   | 5 appels, 64 s perdues
appel_robuste : abandon immédiat : code 401, réessayer ne changera rien | 1 appels, 0 s perdues


**Lecture.** Même résultat (échec), mais le client naïf a frappé 5 fois à la porte et perdu une minute. Le bon
client lit le **code** : 401, 402, 403, 404 demandent une correction humaine, on s'arrête avec un message clair.

Le client complet du projet (`segvet.api.SegformerClient`) combine les quatre protections, et chacune est vérifiée
par un test automatique dans `tests/test_api.py`.

### ✍️ À toi de jouer

Le serveur peut aussi indiquer combien de temps attendre, dans l'en-tête `Retry-After` (en secondes). Modifie `appel_robuste` pour qu'il accepte une liste de couples `(code, retry_after)` et utilise `retry_after` quand il est fourni, l'attente exponentielle sinon. Teste avec `[(429, 7), (200, None)]` : il doit attendre 7 s.

In [5]:
# Ta réponse ici

<details>
<summary><b>💡 Voir une solution</b> (après avoir cherché !)</summary>

```python
def appel_robuste_v2(reponses, essais_max=5):
    attente_totale = 0
    for essai in range(essais_max):
        code, retry_after = reponses.pop(0) if reponses else (200, None)
        if code == 200:
            return f"succès à l'essai {essai + 1}, {attente_totale} s d'attente"
        if code not in A_REESSAYER:
            return f"abandon : code {code}"
        attente = retry_after if retry_after is not None else 2 * 2**essai
        attente_totale += attente
    return "échec"

appel_robuste_v2([(429, 7), (200, None)])   # 'succès à l'essai 2, 7 s d'attente'
```

**Pourquoi ?** le serveur sait mieux que nous quand il sera de nouveau disponible : quand il le dit, on l'écoute. L'attente exponentielle n'est qu'une estimation par défaut.

</details>

## ✅ À retenir

1. Toujours un `timeout` : sans lui, une requête peut bloquer la boucle indéfiniment.
2. On réessaie **seulement** les erreurs passagères (429, 5xx, timeout), avec une attente qui double ou celle de `Retry-After`.
3. 401, 402, 403, 404 : on s'arrête tout de suite avec un message qui dit quoi corriger.
4. On limite soi-même le débit (N requêtes par minute glissante) plutôt que d'attendre les refus du serveur.
5. Une image en échec est **notée** et la boucle continue : on relancera seulement les manquantes.